# 1. Data Processing — UNSW-NB15

Loads raw UNSW-NB15 CSVs using `NUSW-NB15_features.csv` for column names,
cleans and encodes features, scales, computes class weights, and saves train/val/test splits.


In [1]:
import os, random, json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

raw_dir    = os.path.join('..', 'data', 'raw')
splits_dir = os.path.join('..', 'data', 'splits')
models_dir = os.path.join('..', 'models')

os.makedirs(splits_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)

print('raw_dir   =', raw_dir)
print('splits_dir=', splits_dir)
print('models_dir=', models_dir)


raw_dir   = ..\data\raw
splits_dir= ..\data\splits
models_dir= ..\models


## Step 1 — Read column names from NUSW-NB15_features.csv

In [2]:
features_path = os.path.join(raw_dir, 'NUSW-NB15_features.csv')
if not os.path.exists(features_path):
    raise FileNotFoundError(f'NUSW-NB15_features.csv not found in {raw_dir}')

features_meta = pd.read_csv(features_path, encoding='ISO-8859-1')
print(features_meta.head())
print('Columns in features file:', features_meta.columns.tolist())

# Extract ordered feature names from 'Name' column
if 'Name' not in features_meta.columns:
    raise KeyError('Expected column "Name" in NUSW-NB15_features.csv')

col_names = features_meta['Name'].astype(str).str.strip().tolist()
print(f'Loaded {len(col_names)} column names:')
print(col_names)


   No.    Name    Type               Description
0    1   srcip  nominal        Source IP address
1    2   sport  integer       Source port number
2    3   dstip  nominal   Destination IP address
3    4  dsport  integer  Destination port number
4    5   proto  nominal     Transaction protocol
Columns in features file: ['No.', 'Name', 'Type ', 'Description']
Loaded 49 column names:
['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'Label']


## Step 2 — Load and concatenate raw UNSW-NB15_*.csv (no header)

In [3]:
part_files = sorted([
    f for f in os.listdir(raw_dir)
    if f.startswith('UNSW-NB15_') and f.endswith('.csv')
])

if not part_files:
    raise FileNotFoundError('No UNSW-NB15_*.csv files found in data/raw')

print('Files found:')
for f in part_files:
    print(' ', f)

frames = []
for f in part_files:
    path = os.path.join(raw_dir, f)
    tmp = pd.read_csv(path, header=None, names=col_names, low_memory=False)
    print(f'{f}: {tmp.shape}')
    frames.append(tmp)

df = pd.concat(frames, ignore_index=True)

# Normalize column names (lowercase, strip spaces)
df.columns = df.columns.str.strip().str.lower()

print('Combined shape:', df.shape)
print('Columns:', df.columns.tolist())
print(df.head(3))


Files found:
  UNSW-NB15_1.csv
  UNSW-NB15_2.csv
  UNSW-NB15_3.csv
  UNSW-NB15_4.csv
UNSW-NB15_1.csv: (700001, 49)
UNSW-NB15_2.csv: (700001, 49)
UNSW-NB15_3.csv: (700001, 49)
UNSW-NB15_4.csv: (440044, 49)
Combined shape: (2540047, 49)
Columns: ['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'sload', 'dload', 'spkts', 'dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'sjit', 'djit', 'stime', 'ltime', 'sintpkt', 'dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'label']
        srcip  sport          dstip dsport proto state       dur  sbytes  \
0  59.166.0.0   1390  149.171.126.6     53   udp   CON  0.001055     132   
1  59.166.0.0  33661  149.171.126.9   1024   

## Step 3 — Basic cleaning and attack_cat normalization

In [4]:
if 'attack_cat' not in df.columns:
    raise KeyError(f'"attack_cat" not found. Available columns: {df.columns.tolist()}')

# Drop clearly non-informative / leakage-prone columns if present
drop_cols = [c for c in ['id', 'srcip', 'dstip'] if c in df.columns]
if drop_cols:
    df = df.drop(columns=drop_cols)
    print('Dropped columns:', drop_cols)

# Normalize attack_cat text
df['attack_cat'] = df['attack_cat'].astype(str).str.strip()

normalization_map = {
    'Backdoors': 'Backdoor',
    'backdoor': 'Backdoor',
    'backdoors': 'Backdoor',
    'normal': 'Normal',
    'nan': 'Normal',
    '': 'Normal'
}

df['attack_cat'] = df['attack_cat'].replace(normalization_map)

# Replace any remaining empty/NaN with Normal
mask_empty = df['attack_cat'].isin(['', 'nan', 'NaN'])
df.loc[mask_empty, 'attack_cat'] = 'Normal'

df['attack_cat'] = df['attack_cat'].fillna('Normal')

print('attack_cat value counts:')
print(df['attack_cat'].value_counts())


Dropped columns: ['srcip', 'dstip']
attack_cat value counts:
attack_cat
Normal            2218764
Generic            215481
Exploits            44525
Fuzzers             24246
DoS                 16353
Reconnaissance      13987
Analysis             2677
Backdoor             2329
Shellcode            1511
Worms                 174
Name: count, dtype: int64


## Step 4 — Handle missing and infinite values

In [5]:
print('NaN counts per column (non-zero only):')
na_counts = df.isna().sum()
print(na_counts[na_counts > 0])

# Fill NaNs: median for numeric, mode for categorical
for col in df.columns:
    if df[col].dtype == 'object':
        if df[col].isna().any():
            mode_val = df[col].mode().iloc[0]
            df[col] = df[col].fillna(mode_val)
    else:
        if df[col].isna().any():
            med_val = df[col].median()
            df[col] = df[col].fillna(med_val)

# Replace infinities then fill again
df = df.replace([np.inf, -np.inf], np.nan)
for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isna().any():
        med_val = df[col].median()
        df[col] = df[col].fillna(med_val)

print('Total remaining NaNs:', int(df.isna().sum().sum()))


NaN counts per column (non-zero only):
ct_flw_http_mthd    1348145
is_ftp_login        1429879
dtype: int64
Total remaining NaNs: 0


## Step 5 — One-hot encode proto / service / state

In [6]:
cat_cols = [c for c in ['proto', 'service', 'state'] if c in df.columns]
print('Categorical columns to encode:', cat_cols)

if cat_cols:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=False)

print('Shape after one-hot encoding:', df.shape)


Categorical columns to encode: ['proto', 'service', 'state']
Shape after one-hot encoding: (2540047, 208)


## Step 6 — Define X and y, label-encode attack_cat

In [7]:
# Drop binary label column if present (we only use attack_cat for multiclass)
extra_drop = [c for c in ['label'] if c in df.columns]
features_df = df.drop(columns=['attack_cat'] + extra_drop)

y_str = df['attack_cat']

le = LabelEncoder()
y_enc = le.fit_transform(y_str)

print('Classes:', list(le.classes_))
print('Encoded distribution:', dict(zip(*np.unique(y_enc, return_counts=True))))

# Save LabelEncoder and class mapping
joblib.dump(le, os.path.join(models_dir, 'label_encoder.pkl'))
print('Saved label_encoder.pkl')

class_mapping = {int(i): cls for i, cls in enumerate(le.classes_)}
with open(os.path.join(models_dir, 'class_mapping.json'), 'w') as f:
    json.dump(class_mapping, f, indent=2)
print('Saved class_mapping.json')


Classes: ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']
Encoded distribution: {np.int64(0): np.int64(2677), np.int64(1): np.int64(2329), np.int64(2): np.int64(16353), np.int64(3): np.int64(44525), np.int64(4): np.int64(24246), np.int64(5): np.int64(215481), np.int64(6): np.int64(2218764), np.int64(7): np.int64(13987), np.int64(8): np.int64(1511), np.int64(9): np.int64(174)}
Saved label_encoder.pkl
Saved class_mapping.json


## Step 7 — Train / validation / test split (70/15/15, stratified)

In [8]:
X = features_df.copy()
y = y_enc

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)

print('Train:', X_train.shape, ' Val:', X_val.shape, ' Test:', X_test.shape)


Train: (1778032, 206)  Val: (381007, 206)  Test: (381008, 206)


## Step 8 — Feature scaling (StandardScaler fit on train only)

In [9]:
# ── Fix hex-encoded port columns ───────────────────────────────────────────
def coerce_to_numeric(series):
    """Convert hex strings like '0x000c' → int, then to float. Others → NaN."""
    def parse(val):
        try:
            if isinstance(val, str) and val.startswith('0x'):
                return int(val, 16)
            return float(val)
        except (ValueError, TypeError):
            return np.nan
    return series.apply(parse)

# Find all object-dtype columns still in X_train (should be numeric but aren't)
object_cols = X_train.select_dtypes(include='object').columns.tolist()
print('Object-dtype columns that need coercion:', object_cols)

for col in object_cols:
    for frame in [X_train, X_val, X_test]:
        frame[col] = coerce_to_numeric(frame[col])

# After coercion some rows may have NaN — fill with column median
for col in object_cols:
    for frame in [X_train, X_val, X_test]:
        med = frame[col].median()
        frame[col] = frame[col].fillna(med)

# Final check — no object columns should remain
remaining = X_train.select_dtypes(include='object').columns.tolist()
if remaining:
    print('WARNING: still object dtype:', remaining)
else:
    print('All columns are numeric. Safe to scale.')

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

joblib.dump(scaler, os.path.join(models_dir, 'scaler.pkl'))
print('Saved scaler.pkl')


Object-dtype columns that need coercion: ['sport', 'dsport', 'ct_ftp_cmd']
All columns are numeric. Safe to scale.
Saved scaler.pkl


## Step 9 — Compute class weights (softened + normalized)

In [10]:
classes = np.unique(y_train)
raw_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)

# Soften extremes: sqrt, normalize to mean=1, clamp
weights = np.sqrt(raw_weights)
weights = weights / weights.mean()
weights = np.clip(weights, 0.1, 5.0)

print('Raw balanced weights:   ', np.round(raw_weights, 3))
print('Final class weights:    ', np.round(weights, 3))

np.save(os.path.join(models_dir, 'class_weights.npy'), weights)
print('Saved class_weights.npy')


Raw balanced weights:    [9.487900e+01 1.090820e+02 1.553300e+01 5.705000e+00 1.047600e+01
 1.179000e+00 1.140000e-01 1.816000e+01 1.680560e+02 1.457403e+03]
Final class weights:     [1.125 1.206 0.455 0.276 0.374 0.125 0.1   0.492 1.497 4.41 ]
Saved class_weights.npy


## Step 10 — Save splits to data/splits/

In [11]:
feature_names = X_train.columns.tolist()

train_df = pd.DataFrame(X_train_s, columns=feature_names)
train_df['label'] = y_train

val_df = pd.DataFrame(X_val_s, columns=feature_names)
val_df['label'] = y_val

test_df = pd.DataFrame(X_test_s, columns=feature_names)
test_df['label'] = y_test

train_path = os.path.join(splits_dir, 'train.csv')
val_path   = os.path.join(splits_dir, 'val.csv')
test_path  = os.path.join(splits_dir, 'test.csv')

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path,   index=False)
test_df.to_csv(test_path, index=False)

print('Saved:', train_path, train_df.shape)
print('Saved:', val_path,   val_df.shape)
print('Saved:', test_path,  test_df.shape)
print('Done. Notebook 2 (training) can now be run.')


Saved: ..\data\splits\train.csv (1778032, 207)
Saved: ..\data\splits\val.csv (381007, 207)
Saved: ..\data\splits\test.csv (381008, 207)
Done. Notebook 2 (training) can now be run.
